# Scheduled allocation and rebalancing

Milestone 2 uses the same synthetic equity snapshot as the first acceptance study.
This is an offline accounting example, **not historical investment evidence**.
The supplied calendar includes every weekday, even real exchange holidays.

The reusable example function in [scheduled_rebalancing.py](scheduled_rebalancing.py)
materializes the first supplied session in each month after a warm-up. It calls
`rt.inverse_volatility_weights`, `rt.scheduled_rebalance`, `rt.performance`,
`rt.risk_contributions`, and `rt.rolling_risk`; inspect that short script for the
complete public API workflow.

In [ ]:
from pathlib import Path
import sys
import socket
import json
import platform
import importlib.metadata
import polars as pl
import research_toolkit as rt
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd()/"pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/"examples"))
from scheduled_rebalancing import run_example

def no_network(*args, **kwargs):
    raise RuntimeError("This notebook is offline")
socket.create_connection = no_network
socket.socket.connect = no_network

CONFIG = dict(lookback=40, warmup_sessions=60, periods_per_year=252,
              gross_leverage=1.25, max_asset_weight=.4, initial_capital=10_000_000.)
outputs = run_example(**CONFIG)
result, report = outputs["result"], outputs["report"]
print(json.dumps(result.metadata, indent=2))
display(result.targets)

## Information and execution timing

Returns for allocation are raw-price, split-aware total returns with the explicit
analytical assumption `dividend_policy="reinvest_ex_close"`. That analytical index
reinvests entitlement before cash is actually paid. The trading ledger separately
uses actual receivables/payment dates, with cash repaying debt; it never pretends
those dividends are spendable early.

Each decision uses 40 return intervals ending **strictly before** its decision
session. Execution occurs on a later supplied session. Changing future returns
cannot change earlier estimates. All targets must meet the 40% risky-proportion
concentration limit; breaches raise rather than clipping weights. The 1.25× target
leverage is measured against post-cost equity. Between scheduled dates both weights
and leverage drift.

The example uses 5 bps commissions, 2 bps half-spread and 3 bps modeled impact on
actual changed notional. Financing is 2% cash/6% borrowing nominal annually,
ACT/365F, with a 25% research maintenance threshold. `reserve` is the explicit
receivable funding policy; at unlevered targets it can leave risky exposure below
the request to avoid an unstated receivable-funded loan.

In [ ]:
sample_audit = pl.DataFrame([{
    "decision": a.metadata["decision_session"], "sample_start": a.metadata["sample_start"],
    "sample_end": a.metadata["sample_end"], "observations": a.metadata["n_obs"],
} for a in outputs["allocations"]])
display(sample_audit, outputs["allocations"][0].estimates)
assert (sample_audit["sample_end"] < sample_audit["decision"]).all()
display(result.rebalances, result.turnover, result.trades)
assert (result.diagnostics["residual"].abs() <= result.diagnostics["tolerance"]).all()
display(result.diagnostics.group_by("code").agg(pl.col("residual").abs().max()))

## Results and risk diagnostics

Both scenarios use identical initial weights, capital, rates, dates and costs.
One keeps its original holdings; the other executes the dated targets. A complete
report requires `require_complete()`; a failed run is not accepted as a full-period
comparison. Turnover counts purchases plus sales against pre-trade equity, with
entry identified separately.

Covariance risk contributions below estimate the volatility of the final requested
basket using only its pre-decision window. They are not realized P&L attribution.
Rolling risk instead uses realized net returns through each close; it is a reporting
view, not information available for trading at that same close.

In [ ]:
comparison = pl.concat([
    report.summary.with_columns(pl.lit("scheduled").alias("strategy")),
    outputs["hold_report"].summary.with_columns(pl.lit("buy_and_hold").alias("strategy")),
])
display(comparison, outputs["risk"].values, outputs["rolling"].values)
ARTIFACTS = ROOT/"artifacts"/"milestone2"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for name, obj in [("equity", report), ("allocation", report), ("exposures", report),
                  ("turnover", result), ("rolling_risk", outputs["rolling"]),
                  ("risk_contributions", outputs["risk"])]:
    fig, ax = getattr(rt.plots, name)(obj)
    fig.savefig(ARTIFACTS/f"{name}.png", dpi=130)
    display(fig)
    plt.close(fig)
for name in ["daily", "trades", "rebalances", "turnover"]:
    getattr(result, name).write_csv(ARTIFACTS/f"{name}.csv")
environment = {name: importlib.metadata.version(name) for name in
               ("research-toolkit", "polars", "matplotlib", "nbformat", "nbclient", "ipykernel")}
environment["python"] = platform.python_version()
(ARTIFACTS/"environment.json").write_text(json.dumps(environment, indent=2))
print(environment)

## Observations from the inspected default run

The scheduled portfolio completes 198 daily intervals from 2024-04-01 to 2025-01-02.
The ten target baskets include entry and nine subsequent rebalances. After starting
with $10 million, it ends at approximately **$10,701,775.64**, a **7.0178%** compounded
return. Buy-and-hold with the same initial allocation ends at about
**$10,723,002.02** (**7.2300%**). This constructed example provides no evidence that
rebalancing improves a strategy; trading and financing costs are part of both results.

The allocation plot shows discrete target changes and intervening drift. The turnover
plot separates initial funding (about 124.84% of pre-entry capital) from the later
baskets (roughly 10–24% each). At scheduled closes, risky exposure returns to the
requested 1.25× post-cost equity; debt then changes through financing, dividend
payments and later trades. The synthetic January 1 execution is valid in this
supplied calendar, but would require different scheduling on a real exchange.

The rolling-volatility plot begins only after 40 observations. Estimated component
risk contributions are close to, but not exactly, equal: inverse-volatility weights
do not generally equalize covariance-based risk contributions. These estimates
assume fixed exposures and deterministic cash/financing, while the ledger records
actual drift and costs. They answer different questions.

These observations were written after inspecting the default outputs and figures.
Changing the configuration requires new calculations and a fresh interpretation.
No calculation function generates investment conclusions.